In [1]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq zstd pciutils lshw > /dev/null

curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
apt-get install -y -qq nodejs > /dev/null
echo "node $(node -v)"

npm install -g n8n@2 --loglevel=error
echo "n8n $(n8n --version)"

curl -fsSL https://ollama.com/install.sh | sh

wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i cloudflared-linux-amd64.deb > /dev/null
cloudflared --version

node v22.23.4

added 2243 packages in 5m

361 packages are looking for funding
  run `npm fund` for details
n8n 2.35.7
cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)




W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>>

In [2]:
import subprocess, time, os, re, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
ollama_proc = subprocess.Popen(["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

MODEL = "llama3.2"
!ollama pull {MODEL}
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [4]:
# from google.colab import drive

# drive.mount('/content/drive')
# !test -f /content/drive/MyDrive/n8n_backup/n8n_data.tgz && tar xzf /content/drive/MyDrive/n8n_backup/n8n_data.tgz -C /content && echo "restored"

In [5]:
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:5678"],
    stdout=open("/content/cloudflared.log", "w"),
    stderr=subprocess.STDOUT
)

PUBLIC_URL = None

for _ in range(60):
    time.sleep(1)
    with open("/content/cloudflared.log") as f:
        m = re.search(
            r"https://[a-z0-9-]+\.trycloudflare\.com",
            f.read()
        )
    if m:
        PUBLIC_URL = m.group(0)
        break

assert PUBLIC_URL, "Tunnel failed, check cloudflared.log"

env = os.environ.copy()

env.update({
    "N8N_USER_FOLDER": "/content/n8n_data",
    "N8N_HOST": PUBLIC_URL.replace("https://", ""),
    "N8N_PROTOCOL": "https",
    "N8N_PORT": "5678",
    "WEBHOOK_URL": PUBLIC_URL + "/",
    "N8N_EDITOR_BASE_URL": PUBLIC_URL + "/",
    "N8N_PROXY_HOPS": "1",
    "N8N_SECURE_COOKIE": "true",
    "GENERIC_TIMEZONE": "Africa/Cairo",
    "N8N_DIAGNOSTICS_ENABLED": "false",
})

os.makedirs("/content/n8n_data", exist_ok=True)

n8n_proc = subprocess.Popen(
    ["n8n", "start"],
    env=env,
    stdout=open("/content/n8n.log", "w"),
    stderr=subprocess.STDOUT
)

for _ in range(120):
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:5678/healthz",
            timeout=2
        )
        if response.status == 200:
            break
    except Exception:
        time.sleep(2)

print("n8n is up →", PUBLIC_URL)


n8n is up → https://lying-checklist-fit-earlier.trycloudflare.com


In [6]:
!curl -s http://127.0.0.1:11434/api/tags | head -c 300; echo
!tail -n 15 /content/n8n.log
print("Public URL:", PUBLIC_URL)

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-10-31T16:44:53.185973082Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_si
Starting migration ChangeOAuthStateColumnToUnboundedVarchar1763572724000
Finished migration ChangeOAuthStateColumnToUnboundedVarchar1763572724000
Starting migration CreateBinaryDataTable1763716655000
Finished migration CreateBinaryDataTable1763716655000
Starting migration CreateWorkflowPublishHistoryTable1764167920585
Finished migration CreateWorkflowPublishHistoryTable1764167920585
Starting migration AddCreatorIdToProjectTable1764276827837
Finished migration AddCreatorIdToProjectTable1764276827837
Starting migration CreateDynamicCredentialResolverTable1764682447000
Finished migration CreateDynamicCredentialResolverTable1764682447000
Starting migration AddDynamicCredentialEntryTable17646893

In [7]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/n8n_backup
# !tar czf /content/drive/MyDrive/n8n_backup/n8n_data.tgz -C /content n8n_data && echo "backed up"